# 🎬 Conditional Latent Residual Rank Analysis
### Full pipeline — Google Colab (GPU recommended)

**Stages:**
1. Install dependencies
2. Clone / upload source files
3. Configure parameters
4. Extract VAE latents from video
5. Build Raw / Diff / Conditional-Residual populations
6. Spectral analysis (SVD, Effective Rank, Scree Plot)
7. UMAP 2-D visualisation
8. Evaluate predictor (SSIM / PSNR / FID)
9. Browse output artefacts

In [4]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ GPU detected:", result.stdout.strip())
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU.")

✅ GPU detected: Tesla T4, 15360 MiB


## 1 · Install Dependencies

In [5]:
%pip install -q \
    torch torchvision \
    diffusers transformers accelerate \
    opencv-python-headless \
    scikit-learn scikit-image \
    scipy \
    umap-learn \
    matplotlib tqdm Pillow

print("✅ All packages installed.")

✅ All packages installed.


## 2 · Clone or Upload Source Files

**Option A (recommended):** clone from GitHub — edit `REPO_URL` to point at your fork.  
**Option B:** upload the `.py` files manually — uncomment the block in the next cell.

In [6]:
import os, sys

# ── Option A: clone from GitHub ───────────────────────────────────────────────
REPO_URL = "https://github.com/abhiram100/video_compression.git"  # ← your repo
REPO_DIR = "video_compression"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Source files:", os.listdir(REPO_DIR))

# ── Option B: upload .py files manually ──────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # select all five .py files
# if "." not in sys.path: sys.path.insert(0, ".")

FileNotFoundError: [Errno 2] No such file or directory: 'video_compression'

## 3 · Configure Pipeline Parameters

Edit the variables below — every subsequent cell reads from here.

In [ ]:
# ── Video source ──────────────────────────────────────────────────────────────
# Option A: path to a video already on Colab / mounted Drive
# VIDEO_PATH = "/content/my_video.mp4"   # ← change this

# Option B: download a sample clip (uncomment to use)
!wget -q -O /content/sample.mp4 "https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4"
VIDEO_PATH = "/content/sample.mp4"

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/results"

# ── Extraction ────────────────────────────────────────────────────────────────
N_FRAMES    = 300          # total frames to extract
VAE_MODEL   = "stabilityai/sd-vae-ft-mse"
FRAME_SIZE  = 512          # resize each frame to this square before encoding
SAMPLING    = "uniform"    # "uniform" (spread across video) | "dense" (first N frames)

# ── Population building ───────────────────────────────────────────────────────
RIDGE_ALPHA = 1.0          # Ridge regularisation strength
PCA_DIMS    = 256          # PCA dims before fitting Ridge

# ── Spectral analysis ─────────────────────────────────────────────────────────
N_SVD_COMPONENTS = None    # None → min(T, D)-1  (recommended)

# ── UMAP ─────────────────────────────────────────────────────────────────────
UMAP_MAX_POINTS = 2000     # points per population (subsampled for speed)

# ── Evaluation ────────────────────────────────────────────────────────────────
N_VIZ        = 20          # side-by-side comparison panels to save
LATENT_SHAPE = "4,64,64"  # C,H,W — (4,64,64) for SD VAE @ 512 px input
COMPUTE_FID  = True        # set False to skip FID and save ~1 min
DECODE_BATCH = 8           # reduce if you hit GPU OOM during VAE decode

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Config OK — output directory:", OUTPUT_DIR)

## 4 · Step 1 — Extract VAE Latents

Encodes each sampled frame through the SD VAE encoder and saves `latents.npy` — shape `(T, C×H×W)`.

In [ ]:
from extract_latents import extract_latents

latents_path = extract_latents(
    video_path=VIDEO_PATH,
    n_frames=N_FRAMES,
    model_id=VAE_MODEL,
    output_dir=OUTPUT_DIR,
    frame_size=FRAME_SIZE,
    sampling=SAMPLING,
)
print("\n✅ Latents saved to:", latents_path)

## 5 · Step 2 — Build Populations

Generates three matrices and fits + saves the Ridge predictor:
- `raw.npy` — raw latents
- `diff.npy` — linear temporal difference `z_t − z_{t−1}`
- `cond.npy` — conditional residual `z_t − Ridge(PCA(z_{t−1}))`
- `predictor.pkl` — fitted SVD + Ridge (reused in Step 5)

In [ ]:
from build_populations import build_populations

paths = build_populations(
    latents_path=latents_path,
    output_dir=OUTPUT_DIR,
    ridge_alpha=RIDGE_ALPHA,
    pca_dims=PCA_DIMS,
)
print("\n✅ Population files:", {k: os.path.basename(v) for k, v in paths.items()})

## 6 · Step 3 — Spectral Analysis

SVD → normalised eigenvalues → Participation Ratio (Effective Rank) → cumulative variance table.

In [ ]:
from spectral_analysis import run_spectral_analysis
from IPython.display import Image as IPImage, display

results = run_spectral_analysis(
    data_dir=OUTPUT_DIR,
    output_dir=OUTPUT_DIR,
    n_components=N_SVD_COMPONENTS,
)

display(IPImage(os.path.join(OUTPUT_DIR, "scree_plot.png")))

In [ ]:
with open(os.path.join(OUTPUT_DIR, "rank_table.txt")) as f:
    print(f.read())

## 7 · Step 4 — UMAP Visualisation

Projects raw latents and conditional residuals into 2-D.  
Colour + size + opacity all encode temporal progression (early = faint/small → late = bright/large).

In [ ]:
from visualize_umap import run_umap

run_umap(
    data_dir=OUTPUT_DIR,
    output_dir=OUTPUT_DIR,
    max_points=UMAP_MAX_POINTS,
)

display(IPImage(os.path.join(OUTPUT_DIR, "umap_scatter.png")))

## 8 · Step 5 — Evaluate Predictor (SSIM / PSNR / FID)

For every consecutive frame pair:
1. Predict `ẑ_t = Ridge(PCA(z_{t−1}))`
2. Decode both `z_t` (ground truth) and `ẑ_t` (prediction) via VAE
3. Compute per-frame SSIM & PSNR
4. Compute FID via Inception-v3 pool3 features
5. Save side-by-side panels: **Previous | Ground Truth | Predicted**

In [ ]:
from evaluate_predictor import run_evaluation

C, H, W = map(int, LATENT_SHAPE.split(","))

metrics = run_evaluation(
    data_dir=OUTPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_id=VAE_MODEL,
    latent_shape=(C, H, W),
    n_viz=N_VIZ,
    decode_batch=DECODE_BATCH,
    compute_fid_flag=COMPUTE_FID,
)

In [ ]:
with open(os.path.join(OUTPUT_DIR, "eval_metrics.txt")) as f:
    print(f.read())

## 9 · Browse Output Artefacts

In [ ]:
print("── Output files ──────────────────────────────────────")
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort(); files.sort()
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for fname in files:
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"{indent}  {fname}  ({size_kb:.0f} KB)")

In [ ]:
import glob, math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

panels = sorted(glob.glob(os.path.join(OUTPUT_DIR, "eval_frames", "*.png")))
n_show = min(6, len(panels))

if n_show == 0:
    print("No panels found — check Step 5 completed successfully.")
else:
    pick = [int(i) for i in np.linspace(0, len(panels) - 1, n_show)]
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 5 * n_show))
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, pick):
        ax.imshow(mpimg.imread(panels[idx]))
        ax.set_title(os.path.basename(panels[idx]), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Optional: download all results as a zip
# import shutil
# from google.colab import files
# shutil.make_archive("/content/results_export", "zip", OUTPUT_DIR)
# files.download("/content/results_export.zip")